# Abia SARIMA Malaria Intervention-Mix Model

Run this notebook from the project root. The source files are modularised under `src/` so every assumption can be changed without rewriting the full workflow.

**Date alignment:** The 64 observations map exactly to the 64 monthly periods from **January 2021 through April 2026**. No month is inserted or treated as missing. The 24-month forecast covers **May 2026 through April 2028**.

## Purpose

Apply only the intervention package specified for each LGA. Coverage scales linearly from BAU to each target over 12 months. The multiplicative residual-risk layer is applied separately to each forecast case stratum.

In [ ]:
from pathlib import Path
import sys, pandas as pd
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
%cd $PROJECT_ROOT

In [ ]:
from src.data import load_lga_metadata
from src.intervention import evaluate_scenarios
from src.config import OUTPUT_DIR, SCENARIO_COVERAGE, EFFECTIVENESS
meta = load_lga_metadata()
group_forecast = pd.read_csv(OUTPUT_DIR / "forecasts" / "group_SARIMA_forecasts_24m.csv", parse_dates=["date"])

In [ ]:
pd.DataFrame(SCENARIO_COVERAGE).T

In [ ]:
pd.DataFrame(EFFECTIVENESS).T

In [ ]:
monthly_scenarios, cumulative_scenarios = evaluate_scenarios(group_forecast, meta)
cumulative_scenarios.head(12)

In [ ]:
monthly_scenarios.to_csv(OUTPUT_DIR / "tables" / "monthly_intervention_scenarios.csv", index=False)
cumulative_scenarios.to_csv(OUTPUT_DIR / "tables" / "cumulative_intervention_scenarios.csv", index=False)

### Core equation

For group `g`, residual risk under scenario `s` is `R = product(1 - coverage × effectiveness)` across the interventions in that LGA's package that apply to the group. Scenario cases equal BAU forecast cases multiplied by `R_scenario / R_BAU`.